# 실험 제목
- 담당: 김영빈
- 날짜: 26/09/26
- 목적: VL(2만 건 중 실제로는 1만 건) 질문으로 Retrieval 정량 평가 (Hit@K, MRR) + 검색 속도 측정

> 끝나면 결과를 `experiments/LOG.md`에 한 줄 남기기

## 1단계. VL 데이터 파싱

`data/raw/VL_은행|보험|증권`을 파싱한다. KB(document_chunk)는 TL로만 만들었으므로, VL은 "모델이 본 적 없는 질문"으로
평가하기에 적합하다. 평가에는 질문 텍스트와 정답 라벨(category/consulting_topic/qa_topic)만 있으면 되고,
답변 텍스트는 필요 없다.

참고: AI-Hub가 나눠준 라벨링데이터는 TL 8만 + VL 1만 = 9만 건이고, 데이터 스키마 문서의 전체 10만 건과는
차이가 있다(나머지 1만 건은 AI-Hub 자체 비공개 테스트셋으로 추정). 지금 있는 VL 1만 건으로 평가를 진행한다.

In [ ]:
import json
from pathlib import Path

import yaml
from tqdm.auto import tqdm


def find_project_root(start_path: Path) -> Path:
    """현재 위치부터 상위 폴더를 확인해 프로젝트 루트를 찾는다."""
    start_path = start_path.resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / ".git").exists() and (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다. dontalk 저장소 내부에서 노트북을 실행해 주세요.")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

vl_files = sorted(
    p for p in RAW_DIR.rglob("*.json")
    if p.is_file() and p.relative_to(RAW_DIR).parts[0].startswith("VL_")
)
print(f"VL JSON 파일 개수: {len(vl_files):,}")


def parse_vl_record(path: Path) -> dict:
    data = json.loads(path.read_text(encoding="utf-8"))
    src, cons = data.get("source", {}), data.get("consulting", {})
    qa = data["qa_data"][0]
    inp = qa.get("input", {})
    return {
        "source_id": src.get("source_id"),
        "qa_id": qa.get("qa_id"),
        "category": cons.get("consulting_category"),
        "consulting_topic": cons.get("consulting_topic"),
        "qa_topic": qa.get("qa_topic"),
        "question": inp.get("question"),
    }


vl_records = [parse_vl_record(p) for p in tqdm(vl_files, desc="VL 파싱")]
vl_records = [r for r in vl_records if r["question"]]  # 질문 없는 건 제외
print(f"평가 후보 질문: {len(vl_records):,}건")

## 2단계. 데이터 누수(leakage) 체크

Retrieval 평가가 의미 있으려면, 평가에 쓰는 VL 질문의 상담(`source_id`)이 KB(TL)에 하나도 없어야 한다.
같은 상담이 양쪽에 다 있으면 "본 적 있는 걸 다시 찾는" 셈이라 평가가 부풀려진다.

In [ ]:
import pandas as pd

qa_flat = pd.read_json(PROCESSED_DIR / "qa_flat.jsonl", lines=True)
tl_source_ids = set(qa_flat.loc[qa_flat["split"] == "train", "source_id"])
vl_source_ids = {r["source_id"] for r in vl_records}

overlap = tl_source_ids & vl_source_ids
print(f"TL 상담 수: {len(tl_source_ids):,}, VL 상담 수: {len(vl_source_ids):,}")
print(f"겹치는 source_id: {len(overlap):,}건 -> 평가 대상에서 제외")

vl_records = [r for r in vl_records if r["source_id"] not in overlap]
print(f"제외 후 평가 후보 질문: {len(vl_records):,}건")

## 3단계. 평가 질문 샘플링

VL 전체(1만 건) 중 500건을 무작위로 뽑는다. seed를 고정해서 다시 실행해도 같은 샘플이 나오게 한다
(프로젝트 다른 곳에서도 seed=42를 쓰고 있어서 통일).

In [ ]:
import random

random.seed(42)
EVAL_N = 500

eval_set = random.sample(vl_records, EVAL_N)
category_counts = pd.Series([r["category"] for r in eval_set]).value_counts()
print(f"평가 질문 {len(eval_set)}건 샘플링")
print(category_counts)

## 4단계. 검색 + 판정

임베딩 모델과 pgvector에 연결하고(이전 노트북과 동일한 설정), 각 질문을 top-5로 검색한다.
정답 판정은 두 기준으로 계산한다:
- **strict**: `category` + `consulting_topic` + `qa_topic` 3개가 모두 일치해야 정답
- **loose**: `category`만 일치해도 정답

기준을 두 개로 나누는 이유: strict만 보면 너무 엄격해서 실제로 쓸만한 결과도 오답 처리될 수 있고,
loose만 보면 너무 헐거워서(9~5개뿐인 카테고리라 category만 맞히는 건 상대적으로 쉬움) 변별력이 떨어진다.
둘을 같이 보면 어느 정도 수준에서 잘 맞는지 감을 잡을 수 있다.

In [ ]:
import os

import psycopg
from dotenv import load_dotenv
from pgvector.psycopg import register_vector
from sentence_transformers import SentenceTransformer

load_dotenv()
model = SentenceTransformer("dragonkue/snowflake-arctic-embed-l-v2.0-ko")
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True)
register_vector(conn)

TOP_K = 5


def search(question: str, top_k: int = TOP_K):
    query_emb = model.encode(question, prompt_name="query", normalize_embeddings=True)
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT category, consulting_topic, qa_topic
            FROM document_chunk
            ORDER BY embedding <=> %s
            LIMIT %s
            """,
            (query_emb, top_k),
        )
        return cur.fetchall()


results = []
for r in tqdm(eval_set, desc="검색+판정"):
    top_k_labels = search(r["question"])
    strict_ranks = [
        i + 1 for i, (c, t, q) in enumerate(top_k_labels)
        if c == r["category"] and t == r["consulting_topic"] and q == r["qa_topic"]
    ]
    loose_ranks = [i + 1 for i, (c, t, q) in enumerate(top_k_labels) if c == r["category"]]
    results.append({
        "category": r["category"],
        "strict_rank": strict_ranks[0] if strict_ranks else None,   # 처음 정답이 나온 순위 (없으면 None)
        "loose_rank": loose_ranks[0] if loose_ranks else None,
    })

results_df = pd.DataFrame(results)
print(f"검색+판정 완료: {len(results_df)}건")

## 5단계. 지표 계산 — Hit@K, MRR

`rank`가 None이 아니고 K 이하면 Hit. MRR은 순위의 역수(1/rank) 평균, 정답이 없으면 0으로 계산한다.

In [ ]:
def hit_at_k(ranks: pd.Series, k: int) -> float:
    return (ranks.notna() & (ranks <= k)).mean()


def mrr(ranks: pd.Series) -> float:
    return ranks.apply(lambda r: 1 / r if pd.notna(r) else 0.0).mean()


print("=== 전체 (strict: category+topic+qa_topic 모두 일치) ===")
for k in (1, 3, 5):
    print(f"Hit@{k}: {hit_at_k(results_df['strict_rank'], k):.3f}")
print(f"MRR: {mrr(results_df['strict_rank']):.3f}")

print("\n=== 전체 (loose: category만 일치) ===")
for k in (1, 3, 5):
    print(f"Hit@{k}: {hit_at_k(results_df['loose_rank'], k):.3f}")
print(f"MRR: {mrr(results_df['loose_rank']):.3f}")

print("\n=== 카테고리별 Hit@5 (strict) ===")
for category, group in results_df.groupby("category"):
    print(f"{category}: {hit_at_k(group['strict_rank'], 5):.3f} (n={len(group)})")

## 6단계. 검색 속도(latency) 측정

질문 하나를 넣었을 때 "임베딩 + pgvector 검색"까지 실제 사용자가 체감하는 전체 시간을 잰다.
100개 질문으로 평균과 p95(상위 5% 느린 경우)를 확인한다.

In [ ]:
import time

latencies_ms = []
for r in tqdm(eval_set[:100], desc="속도 측정"):
    t0 = time.perf_counter()
    search(r["question"])
    latencies_ms.append((time.perf_counter() - t0) * 1000)

latencies = pd.Series(latencies_ms)
print(f"평균: {latencies.mean():.1f}ms")
print(f"p95: {latencies.quantile(0.95):.1f}ms")
print(f"최대: {latencies.max():.1f}ms")

## 7단계. 증권 카테고리가 왜 약한지 원인 분석

가설 후보:
1. 증권 안에서도 특정 세부 토픽(`qa_topic`)에 실패가 몰려있을 수 있다 (토픽 간 용어가 비슷해서 헷갈림)
2. KB(TL)에 그 토픽 데이터 자체가 적어서 정답 후보가 부족할 수 있다
3. category 판단은 맞는데(loose는 높음) 세부 토픽만 헷갈리는 것일 수 있다

`eval_set`(정답 라벨)과 `results`(판정 결과)를 합쳐서 하나씩 확인한다.

In [ ]:
combined_df = pd.DataFrame([{**r, **res} for r, res in zip(eval_set, results)])
sec_df = combined_df[combined_df["category"] == "증권"]

print(f"증권 평가 질문: {len(sec_df)}건")
print(f"증권 loose Hit@5 (category만): {hit_at_k(sec_df['loose_rank'], 5):.3f}")
print(f"증권 strict Hit@5 (topic까지): {hit_at_k(sec_df['strict_rank'], 5):.3f}")
print("-> loose 는 높고 strict 만 낮으면, 분야는 맞히는데 세부 토픽에서 헷갈리는 것")

## 8단계. 증권 세부 토픽(`qa_topic`)별 성능 + KB 데이터 양

In [ ]:
print("=== qa_topic별 strict Hit@5 (평가셋 기준) ===")
topic_perf = sec_df.groupby("qa_topic").apply(
    lambda g: pd.Series({"n_eval": len(g), "hit@5": hit_at_k(g["strict_rank"], 5)})
)
print(topic_perf.sort_values("hit@5"))

print("\n=== qa_topic별 KB(TL) 문서 수 ===")
kb_topic_counts = qa_flat.loc[qa_flat["category"] == "증권", "qa_topic"].value_counts()
print(kb_topic_counts)

## 9단계. 실제로 뭘 잘못 찾아왔는지 사례 확인

In [ ]:
failures = sec_df[sec_df["strict_rank"].isna()]
print(f"증권 strict 실패(top-5 안에 정답 없음): {len(failures)}건 / {len(sec_df)}건")

sample_failures = failures.sample(min(5, len(failures)), random_state=42)
for _, row in sample_failures.iterrows():
    print(f"\n[질문] {row['question']}")
    print(f"[정답 라벨] {row['category']} / {row['consulting_topic']} / {row['qa_topic']}")
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT category, consulting_topic, qa_topic
            FROM document_chunk
            ORDER BY embedding <=> %s
            LIMIT 5
            """,
            (model.encode(row["question"], prompt_name="query", normalize_embeddings=True),),
        )
        for i, (c, t, q) in enumerate(cur.fetchall()):
            mark = "O" if (c, t, q) == (row["category"], row["consulting_topic"], row["qa_topic"]) else " "
            print(f"  [{mark}] {i+1}위: {c} / {t} / {q}")

## 관찰 / 메모
- VL 1만 건 중 TL과 겹치는 source_id 3건 발견 -> 평가 대상에서 제외 (9,994건 중 500건 샘플링, 카테고리 비율 유지)
- Strict(3개 라벨 모두 일치): Hit@1 0.684 / Hit@3 0.838 / Hit@5 0.878 / MRR 0.763
- Loose(category만): Hit@1 0.914 / Hit@3 0.976 / Hit@5 0.986 / MRR 0.945 -> 메타데이터 기반 라우팅 근거로 쓰기 좋은 수치
- 카테고리별 Hit@5(strict): 보험 0.974 / 은행 0.852 / 증권 0.792 (가장 약함)
- 검색 속도: 평균 25.1ms, p95 29.6ms, 최대 54.9ms -> 실시간 서비스에 충분

### 증권 카테고리 실패 원인 분석 (7~9단계)
- 증권 loose Hit@5 0.948 vs strict Hit@5 0.792 -> 카테고리(분야) 판단은 거의 다 맞고, 세부 qa_topic 구분에서만 정확도 하락
- qa_topic별 편차 큼: 계좌관리(개설/해지등) 0.500(최악) ~ 절세형금융상품 1.000(최고). KB 문서 수와 완전히 비례하지 않아
  단순 데이터 부족보다는 토픽 간 개념 중첩이 더 큰 원인으로 보임
- 실패 사례 확인 결과 2가지 패턴:
  (1) 개념적으로 인접한 토픽끼리 혼동 (증권계좌조회 <-> 계좌관리, 해외주문 <-> 주식주문/증권계좌조회) -- 라벨링 경계 자체가 애매할 수 있음
  (2) 질문에 증권 특화 키워드가 없어 카테고리 경계를 넘어 은행 문서까지 섞여 나오는 경우 (예: "알림 서비스 유무료" 질문)
- 개선 후보(다음 단계): 자주 혼동되는 토픽 쌍 통합 검토, Hybrid Search(BM25 병행)로 키워드 매칭 보강 (기획안 16.2절)
